In [2]:
import cohere
import numpy as np
import pandas as pd
from tqdm import tqdm
import os
from dotenv import load_dotenv

api_key= os.getenv("cohere_api_key")
api_key=api_key
co= cohere.Client(api_key)


In [3]:
text = """
Interstellar is a 2014 epic science fiction film co-written,
directed, and produced by Christopher Nolan.
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain,
Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.
Set in a dystopian future where humanity is struggling to
survive, the film follows a group of astronauts who travel
through a wormhole near Saturn in search of a new home for
mankind.
Brothers Christopher and Jonathan Nolan wrote the screenplay,
which had its origins in a script Jonathan developed in 2007.
Caltech theoretical physicist and 2017 Nobel laureate in
Physics[4] Kip Thorne was an executive producer, acted as a
scientific consultant, and wrote a tie-in book, The Science of
Interstellar.
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in
the Panavision anamorphic format and IMAX 70 mm.
Principal photography began in late 2013 and took place in
Alberta, Iceland, and Los Angeles.
Interstellar uses extensive practical and miniature effects and
the company Double Negative created additional digital effects.
Interstellar premiered on October 26, 2014, in Los Angeles.
In the United States, it was first released on film stock,
expanding to venues using digital projectors.
The film had a worldwide gross over $677 million (and $773
million with subsequent re-releases), making it the tenth-highest
grossing film of 2014.
It received acclaim for its performances, direction, screenplay,
musical score, visual effects, ambition, themes, and emotional
weight.
It has also received praise from many astronomers for its
scientific accuracy and portrayal of theoretical astrophysics.
Since its premiere, Interstellar gained a cult following,[5] and
now is regarded by many sci-fi experts as one of the best
science-fiction films of all time.
Interstellar was nominated for five awards at the 87th Academy
Awards, winning Best Visual Effects, and received numerous other
accolades"""

texts= text.split('.')

texts= [t.strip(' \n') for t in texts]

In [4]:
response= co.embed(
    texts=texts,
    input_type="search_document",

).embeddings

embeds=np.array(response)
print(embeds.shape)

(15, 4096)


In [5]:
import faiss 
dim= embeds.shape[1] #4096
index= faiss.IndexFlatL2(dim)
print(index.is_trained)
index.add(np.float32(embeds))


True


In [6]:
def search(query, num_of_results=3):
    query_embed= co.embed(texts=[query], input_type="search_query").embeddings[0] #.embeddings0 fetches the embedding vector for query 

    distances, similar_items_ids= index.search(np.float32([query_embed]), num_of_results)

    texts_np= np.array(texts)
    results= pd.DataFrame(data={'texts': texts_np[similar_items_ids[0]], 'distance':distances[0]})
#                                            ^^^this is fancy indexing allowing it to fetch the content on the index given
    print(f"Query: '{query}'\nNearest neighbors:")
    
    # Display full sentences without truncation
    for i, (idx, row) in enumerate(results.iterrows()):
        print(f"{i}: {row['texts']} (distance: {row['distance']:.6f})")
    
    return results

In [7]:
query= "how precise was the science?"
results=search(query)
results

Query: 'how precise was the science?'
Nearest neighbors:
0: It has also received praise from many astronomers for its
scientific accuracy and portrayal of theoretical astrophysics (distance: 9659.714844)
1: Interstellar uses extensive practical and miniature effects and
the company Double Negative created additional digital effects (distance: 10615.987305)
2: Caltech theoretical physicist and 2017 Nobel laureate in
Physics[4] Kip Thorne was an executive producer, acted as a
scientific consultant, and wrote a tie-in book, The Science of
Interstellar (distance: 10969.983398)


,texts,distance
0,It has also received praise from many astronom...,9659.714844
1,Interstellar uses extensive practical and mini...,10615.987305
2,Caltech theoretical physicist and 2017 Nobel l...,10969.983398


In [8]:
#we compare semantic search to keyword search
#Lexical search, also known as keywords search, refers to a search algorithm based on the word-level analysis of text
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction import _stop_words
import string

def bm25_tokenizer(text):
    tokenize_doc=[]#empty array or list
    for token in text.lower().split(): #lower case then split on whitespaces
        token= token.strip(string.punctuation) # remove punc from the start and the end of the tokens

        if len(token)>0 and token not in _stop_words.ENGLISH_STOP_WORDS:
            tokenize_doc.append(token)
    return tokenize_doc

In [9]:
tokenized_corpus= []
for passage in tqdm(texts):
    tokenized_corpus.append(bm25_tokenizer(passage))

bm25= BM25Okapi(tokenized_corpus)
def keyword_search(query, top_k=3, num_candidates=15):
    print("Input question:", query)


    bm25_scores= bm25.get_scores(bm25_tokenizer(query))
    top_n=np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
#what argpartition does is it will sort the (num_cands) top scores to the end of the array in any order 
    bm25_hits= [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]

    bm25_hits= sorted(bm25_hits, key=lambda x: x['score'], reverse=True) #sort the top 15 fetched elements, key tells to sort by the score

    print(f"top-3 lexical search (BM25) hits")
    for hit in bm25_hits[0:top_k]:
        print("\t{:.3f}\t{}".format(hit['score'],  #tab space- float point to 3 decimal- tab- {} for string to come(text)
        texts[hit['corpus_id']].replace("\n"," "))) # fetch the content then replace line breaks with space

100%|██████████| 15/15 [00:00<00:00, 109226.67it/s]


In [10]:
keyword_search(query="how precise was the science")

Input question: how precise was the science
top-3 lexical search (BM25) hits
	1.789	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
	1.373	Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar
	0.000	It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine


***not useful^***



RERANKING


In [11]:
query="how precise was the science?"
results= co.rerank(query=query, documents=texts, top_n=3, return_documents=True)
results.results

[RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='It has also received praise from many astronomers for its\nscientific accuracy and portrayal of theoretical astrophysics'), index=12, relevance_score=0.13627496),
 RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='The film had a worldwide gross over $677 million (and $773\nmillion with subsequent re-releases), making it the tenth-highest\ngrossing film of 2014'), index=10, relevance_score=0.038053043),
 RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='Since its premiere, Interstellar gained a cult following,[5] and\nnow is regarded by many sci-fi experts as one of the best\nscience-fiction films of all time'), index=13, relevance_score=0.033415657)]

In [12]:
for idx, result in enumerate(results.results):
    print(idx,result.index, result.relevance_score, result.document.text)

0 12 0.13627496 It has also received praise from many astronomers for its
scientific accuracy and portrayal of theoretical astrophysics
1 10 0.038053043 The film had a worldwide gross over $677 million (and $773
million with subsequent re-releases), making it the tenth-highest
grossing film of 2014
2 13 0.033415657 Since its premiere, Interstellar gained a cult following,[5] and
now is regarded by many sci-fi experts as one of the best
science-fiction films of all time


In [13]:
def keyword_and_rerank(query, top_k=3, num_of_cands=10):
    print("the input question:", query)

    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -num_of_cands)[-num_of_cands:]

    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)

    print(f"top 3- lexical search hits:")

    for hit in bm25_hits[0:top_k]:
        print("\t{:.3f}\t{}".format(hit['score'], texts[hit['corpus_id']].replace("\n", " ")))

    # add reranking
    docs = [texts[hit['corpus_id']] for hit in bm25_hits]
    print(f"\nTop-3 hits by rank-API ({len(bm25_hits)} BM25 hits re-ranked)")
    results = co.rerank(query=query, documents=docs, top_n=top_k, return_documents=True)

    for hit in results.results:
        if hit.document is not None and hasattr(hit.document, "text"):
            print("\t{:.3f}\t{}".format(hit.relevance_score, hit.document.text.replace("\n", " ")))
        else:
            print("\t{:.3f}\t[No text available]".format(hit.relevance_score))

In [14]:
keyword_and_rerank(query="how precise was the science?")

the input question: how precise was the science?
top 3- lexical search hits:
	1.789	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
	1.373	Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar
	0.000	Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects

Top-3 hits by rank-API (10 BM25 hits re-ranked)
	0.032	Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar
	0.029	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
	0.025	Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who

In [54]:
import nltk
# nltk.download("punkt")
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer

model_name=SentenceTransformer("multi-qa-mpnet-base-dot-v1")
docs="""
Interstellar is a 2014 epic science fiction film co-written,
directed, and produced by Christopher Nolan.
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain,
Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.
Set in a dystopian future where humanity is struggling to
survive, the film follows a group of astronauts who travel
through a wormhole near Saturn in search of a new home for
mankind.
Brothers Christopher and Jonathan Nolan wrote the screenplay,
which had its origins in a script Jonathan developed in 2007.
Caltech theoretical physicist and 2017 Nobel laureate in
Physics[4] Kip Thorne was an executive producer, acted as a
scientific consultant, and wrote a tie-in book, The Science of
Interstellar.
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in
the Panavision anamorphic format and IMAX 70 mm.
Principal photography began in late 2013 and took place in
Alberta, Iceland, and Los Angeles.
Interstellar uses extensive practical and miniature effects and
the company Double Negative created additional digital effects.
Interstellar premiered on October 26, 2014, in Los Angeles.
In the United States, it was first released on film stock,
expanding to venues using digital projectors.
The film had a worldwide gross over $677 million (and $773
million with subsequent re-releases), making it the tenth-highest
grossing film of 2014.
It received acclaim for its performances, direction, screenplay,
musical score, visual effects, ambition, themes, and emotional
weight.
It has also received praise from many astronomers for its
scientific accuracy and portrayal of theoretical astrophysics.
Since its premiere, Interstellar gained a cult following,[5] and
now is regarded by many sci-fi experts as one of the best
science-fiction films of all time.
Interstellar was nominated for five awards at the 87th Academy
Awards, winning Best Visual Effects, and received numerous other
accolades"""
docsss=sent_tokenize(docs)

document_embed=model_name.encode(docsss)

query="what "
quembed=model_name.encode(query)


In [55]:
from sentence_transformers import SentenceTransformer, util
top_k=3
retr=util.cos_sim(document_embed, quembed)[0]
top_n=np.argsort(-retr)[:top_k]

for idx in top_n:
    print(f"cos_scores: {retr[idx]:.4f} | text: {docsss[idx]}")

cos_scores: 0.2913 | text: 
Interstellar is a 2014 epic science fiction film co-written,
directed, and produced by Christopher Nolan.


**basic rag**

In [58]:
query="movie cast"
results=search(query)


docs_dict= [{'text': text} for text in results['texts']]
response=co.chat(message=query, documents=docs_dict,
)
print(response.text)


Query: 'movie cast'
Nearest neighbors:
0: It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain,
Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine (distance: 8856.058594)
1: Interstellar is a 2014 epic science fiction film co-written,
directed, and produced by Christopher Nolan (distance: 12761.642578)
2: Caltech theoretical physicist and 2017 Nobel laureate in
Physics[4] Kip Thorne was an executive producer, acted as a
scientific consultant, and wrote a tie-in book, The Science of
Interstellar (distance: 13080.711914)
The cast of Interstellar includes:
- Matthew McConaughey
- Anne Hathaway
- Jessica Chastain
- Bill Irwin
- Ellen Burstyn
- Matt Damon
- Michael Caine


In [61]:
#define query
query= "was the movie a success?"
results=search(query)

docs_dict= [{'text':text}for text in results['texts']]


response=co.chat(message=query, documents=docs_dict)
print(response.text)

Query: 'was the movie a success?'
Nearest neighbors:
0: The film had a worldwide gross over $677 million (and $773
million with subsequent re-releases), making it the tenth-highest
grossing film of 2014 (distance: 7606.263184)
1: It received acclaim for its performances, direction, screenplay,
musical score, visual effects, ambition, themes, and emotional
weight (distance: 8436.433594)
2: It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain,
Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine (distance: 9108.888672)
Yes, the movie was a success. It had a worldwide gross of over $677 million and was the tenth-highest grossing film of 2014. It also received acclaim for its performances, direction, screenplay, musical score, visual effects, ambition, themes, and emotional weight.


In [62]:
# Using Ollama with Llama3
from langchain_ollama import OllamaLLM

# Initialize Ollama with Llama3
llm = OllamaLLM(
    model="llama3:latest", 
    temperature=0.7,
    num_predict=500,  # Max tokens to generate (Ollama parameter)
    num_ctx=2048,     # Context window size (Ollama parameter)
    verbose=False,
)

In [ ]:
from langchain.embeddings.huggingface import HuggingFaceBgeEmbeddings
#initialize th embedding model
embedding_model= HuggingFaceBgeEmbeddings(model_name='thenlper/gte-small')

from langchain.vectorstores import FAISS #faiss for fast search 
db=FAISS.from_texts(texts, embedding_model)

/var/folders/ql/hdbyk3lx6vq0x6fs9227fc_m0000gn/T/ipykernel_21183/3093941218.py:3: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model= HuggingFaceBgeEmbeddings(model_name='thenlper/gte-small')
/Users/abhimanyu/Desktop/geospacy/venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
from langchain import PromptTemplate
#define. a prompt template, which is very crucial to be in correct format for rag pipelines
template= """<s><SYS>just answer the question, no fluff[INST][/SYS]
Relevant information:
{context}
Provide a concise answer the following question using the
relevant information provided above:
{question}[/INST]"""

prompt= PromptTemplate(template=template, input_variables=["context", "question"])

from langchain.chains import RetrievalQA

rag= RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=db.as_retriever(),
    chain_type_kwargs={'prompt': prompt},
    verbose=True
)

In [70]:
rag.invoke("cast of the movie?")



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'cast of the movie?',
 'result': 'The cast of the movie includes:\n\n* Matthew McConaughey\n* Anne Hathaway\n* Jessica Chastain\n* Bill Irwin\n* Ellen Burstyn\n* Matt Damon\n* Michael Caine'}

if a query is too verbose, it needs query rewriting in order to make retrieval possible




It evaluates results along four axes:
Fluency
Whether the generated text is fluent and cohesive.

Perceived utility
Whether the generated answer is helpful and informative.
Citation recall
The proportion of generated statements about the external world that are
fully supported by their citations.
Citation precision
The proportion of generated citations that support their associated
statements.

In [71]:
rag.invoke("did users like this movie?")



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'did users like this movie?',
 'result': 'Yes, according to the text, the movie received acclaim for various aspects, including performances, direction, screenplay, and more.'}